In [1]:
import pandas as pd
import json
import yaml
import seaborn as sns
from pathlib import Path

In [2]:
# folder = Path('..', 'outputs', '005.experiment')
folder = Path('..', 'outputs', '005A01.experiment')

experiments = [ p for p in folder.iterdir()
                if p.is_dir() and Path(p, 'eval_results.json').exists() ]

for p in experiments: print(p.name)

6844381976582f7adf06ba5e3544388f
e41e94824b2be71f70e3e451b582fdb0
71b03bb0ef1a36d0806b2d0865686c9d


In [3]:
def get_params(dir):
    params_file = Path(dir, 'parameters.yaml')
    parameters = yaml.safe_load(params_file.read_text())
    # parameters['folder'] = dir.name
    return parameters

def read_json_results(folder, base_file):
    file = Path(folder, base_file)
    results = json.loads(file.read_text())
    results['folder'] = folder.name
    return results

df_params = pd.DataFrame([ get_params(p) for p in experiments ])
df_eval = pd.DataFrame([read_json_results(e, 'eval_results.json') for e in experiments])
df_valids = pd.DataFrame([read_json_results(e, 'valid_results.json') for e in experiments])

df = pd.merge(df_params, df_eval, left_on='_hash_id', right_on='folder')

teacher_keys = df['teachers_keys'].loc[0] # ['t5', 'llama']

df = pd.concat([
    df,
    df['teachers_weights'].apply(pd.Series, index=teacher_keys)
], axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 36 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   _hash_id                 3 non-null      object 
 1   _timestamp               3 non-null      object 
 2   batch_size               3 non-null      int64  
 3   bf16                     3 non-null      bool   
 4   dataset                  3 non-null      object 
 5   eval_steps               3 non-null      int64  
 6   experiment               3 non-null      object 
 7   from_pretrained          3 non-null      object 
 8   generation_max_length    3 non-null      int64  
 9   grad_steps               3 non-null      int64  
 10  local_rank               3 non-null      int64  
 11  logging_strategy         3 non-null      object 
 12  lora_rank                3 non-null      int64  
 13  lr                       3 non-null      float64
 14  max_input_length         3 non

In [4]:
df_valids

,epoch,eval_accuracy,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_token_accuracy,folder
0,33.817907,0.740,0.020628,112.1960,4.456,0.499,0.81496,6844381976582f7adf06ba5e3544388f
1,12.704174,0.758,0.015076,111.6226,4.479,0.502,0.81450,e41e94824b2be71f70e3e451b582fdb0
2,25.362976,0.766,0.018252,109.7969,4.554,0.510,0.81488,71b03bb0ef1a36d0806b2d0865686c9d


In [5]:
df[
    ['dataset', '_timestamp', 'batch_size', 'lr', 'lora_rank', 'grad_steps', 'eval_accuracy', 'eval_token_accuracy', 'eval_loss']
].sort_values(by='eval_accuracy', ascending=False)

,dataset,_timestamp,batch_size,lr,lora_rank,grad_steps,eval_accuracy,eval_token_accuracy,eval_loss
2,obqa,2025-11-21T01:29:04.718435,9,0.0005,16,6,0.728,0.91614,0.022944
1,obqa,2025-11-20T17:23:58.680946,9,0.0005,16,3,0.722,0.91558,0.018841
0,obqa,2025-11-21T11:57:48.291849,9,0.0005,16,8,0.692,0.91626,0.025085
